# The Impact of M&A on Inventor Mobility and Innovation

## A. Overview of the GitHub package

This notebook is a companion to the construction and analysis pipeline in the repository.  It explains how the data are built and how the main empirical designs work, presents a headline story, and distinguishes robust findings from fragile or method-dependent ones.

The GitHub version of the project separates the work into:

- **construction scripts**: build cleaned firm-year and inventor-year panels,
- **analysis scripts**: baseline DiD, event studies, heterogeneity, and advanced estimators,
- **output folders**: regression tables, coefficient plots, and summary CSVs,


### Main empirical findings story

M&A appears to reorganize innovation.  Acquiror inventors are reoriented toward exploration around the deal window, with mobility patterns consistent with short-run reassignment rather than a persistent mobility increase.  Target-side and firm-level outcomes show substantial disruption and innovation output decline, but their event-study paths also reveal pre-period movement and the results are more fragile across methods, so those results should be interpreted more cautiously as disruption and selection evidence rather than the clean causal evidence.


### Construction summary

The data construction for the project is intended to provide users and researcher a useful guide on how to work with inventor-level innovation data.  The project starts from raw patent, accounting, and deal records and builds a linked set of firm-level and inventor-level panels that can speak to both firm- and inventor-level outcomes around M&As.  In the completed construction run, the inventor M&A event-study panel is fully balanced over the `[-5, +5]` window, with stable treated and control row counts at each relative year.  Treated inventors are also matched to control units with reasonably good pre-period balance at `t = -1`. 


## B. Construction

### Overview and design logic

A major part of the value of this project is the construction itself.  It combines different data sources which provide useful information into usable research datasets:

- **PatentsView** provides patent, inventor, assignee, location, CPC, and citation information.
- **External patent-quality data** add KPSS-based measures such as `xi_real` and citation fields.
- **Compustat** provides firm fundamentals.
- **WRDS link tables** connect Compustat firms to CRSP/market identifiers such as `permco`.
- **SDC M&A data** provide acquisition and target events.

### Construction pipeline at a glance

The construction code is organized into eight sequential modules plus one runner script.

| File | Main purpose | Main outputs |
|---|---|---|
| `run_construction.py` | Executes all construction modules in order | End-to-end pipeline |
| `01_setup_helpers.py` | Paths, imports, global helpers, data download helper | Shared environment |
| `02_patent_panel_construction.py` | Builds the core patent-inventor-firm dataset and patent-level measures | `pat_inv_firm_df.pkl` and intermediate patent objects |
| `03_exploration_exploitation.py` | Adds exploration/exploitation metrics | enriched patent-inventor-firm file |
| `04_mobility_and_mover_metrics.py` | Detects inventor moves and builds move-related performance objects | mover event and move-performance files |
| `05_technology_similarity.py` | Computes technology proximity and rolling similarity measures | event-based and rolling similarity outputs |
| `06_firm_fundamentals.py` | Builds Compustat-based firm fundamentals and links them to `permco` | linked Compustat panel |
| `07_linking_and_manda.py` | Cleans and links M&A deals and adds pre-deal technology similarity | `manda.pkl` |
| `08_final_panels.py` | Produces the final firm-year, firm-event, inventor-year, and inventor-event panels | final analysis panels |



### Step 0. Environment, paths, and helper infrastructure

The setup module does four jobs:

1. loads the core Python stack,
2. defines the project paths,
3. validates that required local inputs exist,
4. provides a helper to download PatentsView files on demand.

#### Main libraries used

The construction is fundamentally a Python data engineering and empirical research pipeline.  The main tools are:

- **pandas** and **NumPy** for table manipulation and vectorized operations,
- **scikit-learn** for matching support and nearest-neighbor logic,
- **tqdm** for progress monitoring in long-running loops,
- **requests** and **zipfile** for downloading and unpacking PatentsView files,
- **collections.Counter / defaultdict / deque** for efficient rolling technology-history objects.


The project also depends on several local raw-data directories, thus the setup code defines a strict path check, to avoid the code failing after several hours of runtime:

```python
def assert_required_paths_exist():
    required_paths = [
        BASE_PROJECT_PATH,
        RAW_DATA_PATH,
        FINANCIAL_DATA_PATH,
        MANDA_DATA_PATH,
        LINKTABLE_CSV,
    ]
    for path in required_paths:
        if not os.path.exists(path):
            raise FileNotFoundError(f"Required path does not exist: {path}")
```



### Step 1. Build the core patent-inventor-firm dataset

 Everything dataset constructed in later steps depends on a clean patent-level file that links:

- a patent,
- its inventor(s),
- the public firm to which it can be assigned,
- the patent's technological and quality characteristics.


#### Raw patent inputs

The construction draws the following core tables from PatentsView:

- inventor-patent links,
- assignee-patent links,
- inventor locations,
- CPC technology classifications,
- patent-to-patent citations,
- patent issue dates,
- application filing dates.

The code begins with a cache-first wrapper, so the core patent datasets are only loaded from PatentsView if they are not already stored locally:


#### Cleaning 

Several cleaning steps are important to construct meaningful data.

- **Keep business assignees and the primary assignee**
    The assignee file is filtered to:

    ```python
    assignee_df = assignee_df[
        (assignee_df['assignee_type'] == 2) &
        (assignee_df['assignee_sequence'] == 0)
    ].copy()
    ```

    This means the construction is intentionally focused on the primary business assignee rather than all possible assignee relationships.  The downstream goal is to connect patents to publicly listed firms in a way that is stable enough for event-study analysis and avoid processing of large numbers of patent data that can be excluded early on.

- **Use filing year rather than issue year**
    The code merges issue dates and application dates and defines.  Filing dates are closer to the time of the actual inventing than grant dates, which can be delayed by the examination processes.

- **Retain detailed CPC information**
    The code keeps CPC subclasses and groups, using primary CPC assignments for some tasks and full CPC combinations for novelty construction later.


#### External data merge

Two external research datasets are merged with the patent data.

- **KPSS patent-quality data**
    The construction merges to the PatentsView data patent-level innovation value `xi`, citation information, and links to `permco` firm identifiers from the replication dataset of [Kogan, Papanikolaou, Seru, and Stoffman (2017) (KPSS)](https://github.com/KPSS2017/Technological-Innovation-Resource-Allocation-and-Growth-Extended-Data).  KPSS data are quickly becoming a standard data source for innovation research and allow the project to easily identify patents assigned to publicly-listed companies.

- **State-level noncompete enforceability**
    The code merges the state-year noncompete enforcement score from [Johnson, Lavetti and Lipsitz (2024)](https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/37A0L2) using inventor state and filing year from PatentsView.  The project is about inventor mobility, so including a local legal environment relevant to mobility is economically meaningful.



#### Patent-level quality measures

The construction build several patent-quality measures, which are all well established in the innovation literature.

- **Forward citations**
    First, unconditional forward citations are computed from citation data.  Then the code constructs class-year normalized bins based on CPC subclass and filing year:

    ```python
    stats = df.groupby(['filing_year', 'cpc_subclass'])['forward_citations']           .quantile([0.90, 0.99]).unstack()
    ```

    Raw citations are noisy across technology classes and cohorts.  Ranking patents within technology-year cells makes the quality proxy more comparable.


- **KPSS-based citation bins**
    The same binning logic is repeated for the KPSS citations, reducing the dependence on a single citation measure from PatentsView.

- **Patent novelty**
    The novelty measure is based on new combinations of CPC classes, following the logic of recombinatorial innovation.  The key idea is the following:

    - represent a patent as a set of technology classes,
    - enumerate all within-patent class pairs,
    - identify whether each pair is new in the historical record,
    - summarize the share of new combinations.

    The number of new combinations of patent class directly measures whether the patent creates knowledge that combines different technology fields in a novel way relative to prior art.

- **Citation-link measures**
    The pipeline also builds:

    - backward citations,
    - self-citations at the firm level.

    Backward citations help to measure if an invention is in a more mature technological area (requiring more prior art to be cited), and self-citations measure whether firms explore new technological areas or focus on developing in their previous areas of expertise.  


#### Final patent-inventor-firm object

After all merges, the project creates the core patent-level research dataset `pat_inv_firm_df` and then adds inventor career features such as:

- first filing year,
- first CPC field,
- inventor age measured as years since first filing,
- indicators for whether a patent stays close to the inventor's original field, i.e., the first CPC field.

Thus, we have a single micro-level innovation dataset from which the downstream panels can be constructed.



### Step 2. Construct exploration and exploitation measures

For each patent, the code builds the set of CPC subclasses and compares that set to the recent knowledge base of:

- the patent's inventors,
- the firm the patent is assigned to.

The knowledge base is defined using a rolling five-year window.  We can then define technology exploration as the share of CPC subclasses that the inventor/firm have not patented in recently.

A simplified version of the logic is:

```python
for row in patent_level.itertuples(index=False):
    inv_knowledge = union_of_recent_cpcs_for_inventors
    firm_knowledge = union_of_recent_cpcs_for_firms

    exp_inv = 1 - overlap(current_cpcs, inv_knowledge) / len(current_cpcs)
    exp_firm = 1 - overlap(current_cpcs, firm_knowledge) / len(current_cpcs)
```

A patent is thus classified as more exploratory when it uses CPC classes that are less represented in the inventor's or firm's recent history.  This is an intuitive way to measure movement into less familiar technologies, which could be a reasonable effect following an acquisition as, for example, inventor teams are combined or firms change innovation focus.  The key intuition is that mergers may reshape what inventors and firms work on even when the total number of patents does not change much.


## Step 3. Identify inventor mobility events and build mover metrics

Inventor mobility is directly measured and then connected to post-M&A firm outcomes.

### Strict move identification

The move definition is intentionally conservative.  The code first restricts attention to inventors with at least five patents:

```python
min_patents = 5
prolific_inventors = inventor_counts[inventor_counts >= min_patents].index
```

Then it defines a move only when there is a stable firm sequence around the transition:

- two patents before the move at the old firm,
- the first patent at the new firm,
- two subsequent patents at the new firm.

The core rule is:

```python
is_move = (
    (career_df['permco'] != career_df['prev_permco']) &
    (career_df['prev2_permco'] == career_df['prev_permco']) &
    (career_df['next_permco'] == career_df['permco']) &
    (career_df['next2_permco'] == career_df['next_permco'])
)
```

### Why this strict rule is useful

Patent assignment data can be noisy.  A single assignee switch may reflect legal assignment timing, joint work, or temporary data noise rather than a real labor-market transition.  The stricter move rule sacrifices some sample size.

### Pre/post mover performance

Once moves are defined, the code builds five-year pre-move and post-move performance windows and aggregates inventor outcomes such as:

- patent counts,
- citations,
- `xi_real`,
- novelty,
- backward and self-citations,
- exploration and exploitation,
- team size.

This produces both:
- a long panel with one row per inventor-move-period, and
- a wide format that makes change calculations straightforward.

### Benchmarking movers relative to peers

The project also compares movers to peers at origin and destination firms.  That is a subtle but valuable addition because it helps distinguish:

- whether firms are losing unusually important inventors,
- whether incoming inventors look stronger or weaker than incumbent peers,
- whether post-move performance changes suggest integration frictions or gains.


## Step 4. Measure technology similarity around moves and deals

The project next constructs technology-similarity objects that help interpret whether mobility and M&A connect technologically related or unrelated domains.

### Event-based proximity around inventor moves

The first routine compares technology vectors before and after a move for:

- the inventor,
- the origin firm,
- the destination firm.

Vectors are represented as `Counter` objects over CPC subclasses, and similarity is measured with cosine similarity:

```python
def _calculate_cosine_similarity(vec1, vec2):
    dot_product = sum(vec1.get(k, 0) * vec2.get(k, 0) for k in all_keys)
    norm1 = sqrt(sum(v**2 for v in vec1.values()))
    norm2 = sqrt(sum(v**2 for v in vec2.values()))
    return dot_product / (norm1 * norm2)
```

This yields quantities such as:

- inventor pre/post self-similarity,
- inventor similarity to origin firm,
- inventor similarity to destination firm,
- origin-destination firm similarity.

### Rolling annual similarity

The second routine creates a rolling annual similarity measure, comparing a current year's technology vector with the preceding five-year technology history.  This is helpful because it turns technology direction into a panel variable rather than a one-time event summary.



## Step 5. Build firm fundamentals from Compustat and link them to market identifiers

The firm-side empirical analysis needs standard controls and heterogeneity variables describing firm size, profitability, leverage, liquidity, investment, and financial constraints.

### Sample construction in Compustat

The code filters Compustat to a standard industrial sample:

- excludes financials, utilities, and public sector entities,
- keeps standardized, consolidated, domestic statements,
- removes clearly invalid negative values for core accounting variables.

It then defines the analysis year as:

- same calendar year if the fiscal year ends in June to December,
- previous calendar year otherwise.

That aligns fiscal records more closely with the year to which they economically belong.

### Feature definition

The module then constructs a broad set of variables, including:

- size: `log_at`, `log_sale`, `log_mv`,
- profitability: `roa`, margins, operating profitability,
- growth and valuation: `sale_growth`, `tobinsq`, `market_to_book`,
- financing and liquidity: leverage, cash, interest coverage,
- investment and composition: capital expenditures, R&D intensity, intangible assets,
- payout and repurchases,
- financial constraints indices such as **Whited-Wu** and **Hadlock-Pierce**.

This creates the economic state variables needed to:
- control for confounding firm conditions,
- define heterogeneity splits,
- interpret M&A effects in a richer organizational context.

The code also downloads CPI data from FRED and uses it to build real assets and real R&D variables. 

Finally, the link of Compustat data to the patent side is done using the CRSP-Compustat link table and the `percmo` identifier.  The code keeps primary, validated links and restricts them to valid date ranges.

```python
compustat_core_linked = compustat_core_df.merge(
    linktable[['gvkey', 'permco', 'linkdt', 'linkenddt']],
    on='gvkey', how='inner'
)
compustat_core_linked = compustat_core_linked[
    (compustat_core_linked['datadate_dt'] >= compustat_core_linked['linkdt']) &
    (compustat_core_linked['datadate_dt'] <= compustat_core_linked['linkenddt'])
]
```



## Step 6. Clean and link M&A deals

The M&A module transforms raw SDC transaction records into a deal panel that can be merged into firm-year and inventor-year data.

### Deal cleaning choices

Several choices are worth highlighting.

#### 1. Announcement timing is central
The project uses announcement-year timing for the event-study setup. Announcement are typically the point at which firms, investors, and employees expect that the merger will complete and innovation behavior adjusts.

#### 2. Link both target and acquiror through CUSIP-6 and valid date windows
The code truncates CUSIPs to six digits (identifying the issuing firm) and merges both sides of the deal through the link table.  It then keeps only observations for which the announcement date falls within the valid link ranges for both firms.

#### 3. Restrict deal outcomes
The code keeps completed and withdrawn deals and explicitly creates a failed-merger indicator.  Failed deals can later be used either as controls, robustness checks, or for identification strategies.

### Deal-level variables

The M&A panel retains:

- acquiror and target identifiers,
- transaction value,
- ownership percentages,
- indicators based on industry codes whether the deal was diversifying,
- deal outcome and failure status.

### Pre-deal technology similarity between target and acquiror

The code also adds a pre-deal technology similarity measure, built from five-year patent portfolios of the target and acquiror.  This helps characterize whether deals combine related or more distant technology portfolios.



## Step 7. Assemble the final analysis panels

The final module turns the intermediate construction objects into the panels used for the empirical analysis.

This is the point where the project shifts from **data integration** to **econometric design**.

### 7.1 Firm-year panel

The code first aggregates patent outcomes to the `permco × year` level:

```python
firm_year_patent_metrics = pat_inv_firm_df.groupby(['permco', 'filing_year']).agg(
    total_patents=('patent_id', 'nunique'),
    cites=('cites', 'sum'),
    xi_real=('xi_real', 'sum'),
    ...
).reset_index()
```

It then merges in:

- rolling firm technology similarity,
- Compustat fundamentals,
- inventor arrival and departure measures,
- relative quality of arriving and departing inventors,
- M&A event counts and deal-value measures.

The result is a firm-year panel that simultaneously describes:
- innovative output,
- technology direction,
- labor flows,
- financial conditions,
- M&A exposure.

### 7.2 Firm-year M&A event-study panel

The next object is a firm-year event-study panel centered on the **closest deal year** within a ±5-year window.

A nice feature is that the code avoids `merge_asof` and instead computes previous and next deal indices explicitly with `np.searchsorted`, making the logic more transparent and less fragile.

The panel then defines:

- `ma_deal_role` = acquiror, target, or no recent M&A,
- `years_from_ma_deal`,
- deal value,
- failed-merger flag,
- other-party identifier,
- pre-deal technology similarity.

This object is the main firm-level treatment panel used for DiD and event-study work.

### 7.3 Inventor-year panel

The inventor-year panel is built for inventors who ever patent at firms in the analysis universe. The code creates:

- annual inventor outcome measures,
- zero-filled years for no-patenting observations within the inventor career span,
- annual employer assignment,
- move-year indicators,
- M&A context of the assigned employer.

This converts sparse patent observations into a panel suitable for longitudinal analysis of inventor behavior.

### 7.4 Inventor-event panel with matched controls

The most sophisticated construction step is the inventor M&A event-study panel.

The logic is:

1. identify treated inventor-deal events,
2. keep only events feasible over the full event window,
3. construct matched control pseudo-events using inventor characteristics at `t = -1`,
4. expand each event into a balanced `[-5, +5]` relative-year grid,
5. merge annual inventor outcomes back onto the event grid.

The matching features include variables such as:

- inventor age,
- cumulative patents,
- cumulative citations,
- first CPC subclass.

The code supports propensity-score-based matching with exact matching on selected categorical fields.  This tries to build a credible inventor-level counterfactual while keeping the event-study object balanced and interpretable.

### Why these final panels are the right endpoint

By the end of the pipeline, the project has produced four conceptually distinct but connected research objects:

1. **firm-year panel** for longitudinal organizational outcomes,
2. **firm event-study panel** for treatment-timing analysis,
3. **inventor-year panel** for individual-level longitudinal behavior,
4. **inventor-event panel** for matched event-study analysis.


## Construction diagnostics: what the final balancing tests imply

The completed construction run also produces useful diagnostics for the inventor M&A event-study panel.  These are worth discussing explicitly because they speak to the credibility of the matched design that underlies the inventor-side analysis.

### Event-time balance and panel integrity

The inventor event-study panel contains **8,243,257 rows and 48 columns**, with **441,242 balanced inventor-events out of 441,242 total inventor-events** over the full `[-5, +5]` window.  This confirms that the event-panel construction worked as intended.

In practice, that means every retained inventor-event contributes the full eleven relative years required for the baseline event-study design.  The reported row counts by relative time are perfectly stable across the window:

- treated rows per event time: **210,521**
- control rows per event time: **538,866**

That stability is exactly what one hopes to see in a stacked matched event-study panel and suggests that the final inventor-event object is structurally well behaved for downstream estimation.

### What the balance checks say

At `t = -1`, the matched treated and control groups are reasonably close on the main inventor characteristics used for matching and interpretation:

- `inventor_age`: treated = **10.531**, control = **9.952**, SMD = **0.098**
- `cum_patents`: treated = **9.776**, control = **7.631**, SMD = **0.144**
- `cum_cites`: treated = **346.441**, control = **197.801**, SMD = **0.129**
- `total_patents`: treated = **0.970**, control = **0.728**, SMD = **0.105**
- `cites`: treated = **25.157**, control = **11.824**, SMD = **0.071**

The overall picture is encouraging rather than perfect.  Standardized mean differences (SMD) around `0.07` to `0.14` indicate that the matching substantially improves comparability, but does not eliminate all pre-existing differences between treated inventors and controls.  In particular, treated inventors remain somewhat more established and somewhat more productive than controls even before the event.  That is not surprising in this setting as larger firms that typically have more productive inventors also are more likely to engage in M&A activity.  

The matched design creates a more credible counterfactual than an unmatched comparison, while still leaving room for residual selection on inventor quality or career trajectory.  That is why the project benefits from advanced staggered-treatment estimators such as Sun–Abraham and Borusyak–Jaravel–Spiess.

### Control reuse: acceptable, but worth noting

The diagnostics also show substantial reuse of matched controls:

- unique control-event matches: **487,373**
- share of controls reused more than once: **0.779**
- maximum reuse count: **132**

This is not a construction failure.  It is a common consequence of nearest-neighbor matching in a large treated-event sample, especially when the design tries to keep controls comparable on age, cumulative productivity, and technological background.  Some control inventors are simply very attractive matches for many treated events.

The raw row count in the final panel is very large, but effective independent variation is somewhat smaller than that row count might suggest because certain controls appear repeatedly.  In practical terms, this does not invalidate the design, but it does strengthen the case for conservative inference and for reading the matched sample as a tool for improving comparability rather than a guarantee of perfect one-to-one counterfactual assignment.

### Bottom line on the construction diagnostics

Taken together, the final diagnostics support the view that the inventor event-study panel is **successfully constructed, balanced in event time, and reasonably well matched on observables**, while still reflecting the familiar empirical reality that treated inventors are somewhat positively selected and that donor controls are reused heavily.


## Analysis: method inventory, headline evidence, and interpretation

The analysis has two goals.  First, it asks whether M&A changes innovation outcomes after the deal.  Second, it asks **where the change occurs**: at the aggregate firm level, at the inventor level, through inventor mobility, or through the direction of inventive search.

The headline result is not that M&A mechanically increases innovation.  A more careful interpretation is that M&A changes the **organization of inventive labor**.  Acquiror inventors show evidence of reorientation: more exploration, more mobility, and dynamic post-deal output changes.  Target inventors show a more disruptive pattern: baseline estimates point to lower patenting and citations, while some dynamic target estimates are less stable and should be interpreted cautiously.

### Note on visualization

Most figures in this notebook are already produced by the analysis scripts under `Model_outputs/Plots`; stacked synthetic-control plots are written under `Model_outputs/advanced`.  To render the notebook without changing the analysis code, copy or sync those plots into the repository-level `figures/` folder while preserving subfolders.

```bash
rsync -av "$MODEL_OUTPUTS/Plots/" "figures/"
rsync -av "$MODEL_OUTPUTS/advanced/" "figures/advanced/"
```

The analysis sections below include the most useful figure slots.  If a linked figure does not render yet, it means the corresponding PNG has not been copied into `figures/` or has not yet been generated by the analysis run.

## 1. Baseline empirical design

The core specification is a two-way fixed-effects DiD model estimated separately for acquiror and target events:

$$
Y_{it} = \beta \cdot \text{PostTreat}_{it} + X_{it-1}'\delta + \alpha_i + \lambda_t + \varepsilon_{it}.
$$

Here $Y_{it}$ is an inventor-year or firm-year outcome, $\alpha_i$ absorbs time-invariant unit differences, $\lambda_t$ absorbs common year shocks, and $X_{it-1}$ contains lagged controls where available.  The coefficient $\beta$ is the average post-M&A difference between treated units and matched or comparison controls, conditional on unit and year fixed effects.

The corresponding event-study specification replaces the single post-treatment indicator with relative-year indicators to the deal-year $G_i$:

$$
Y_{it} = \sum_{k \neq -1} \beta_k \cdot 1\{t-G_i=k\}\times \text{Treated}_i + X_{it-1}'\delta + \alpha_i + \lambda_t + \varepsilon_{it}.
$$

The omitted period is $k=-1$, so each $\beta_k$ is interpreted relative to the year immediately before the deal.  The implementation is deliberately compact: the same fixed-effects wrapper is reused across firm and inventor designs, while role-specific runners construct the appropriate acquiror or target panels.

### Baseline event-study graphics used throughout

The baseline event-study plots show both **identification diagnostics** and **economic timing**.  In each plot, pre-period coefficients should be read as a parallel-trends diagnostic, i.e., whether the identifying assumption that the merger had an effect on the relevant outcomes is credible.  Post-period coefficients show whether the effect is immediate, delayed, persistent, or transitory.

The active plotting routine in `firm_analysis.py` saves these figures as:

```text
es_{role_tag}_{outcome}.png
```

For the inventor-year preferred specification, including firm- and inventor-level controls, the key `role_tag` values are:

```text
inv_year_acquiror_vs_control_nn1_x_firm
inv_year_target_vs_control_nn1_x_firm
```


## 2. Why the inventor-year panel is the headline layer

The inventor-year panel directly observes the mechanism that motivated the project: inventor behavior around M&A.  It separates changes in:

- **productivity**: `total_patents`, `cites`, and `xi_real`,
- **search direction**: `exploration_inv` and `exploitation_inv`,
- **mobility**: `is_move_year`,
- **novelty**: `novelty_score_group`.

The preferred specification is `nn1_x_firm`: nearest-neighbor or propensity-score matched inventor control groups, inventor-level controls, and lagged firm controls.  

The acquiror inventor-year sample is large: **4,105,277 rows**, **119,682 inventors**, and **2,779 firms**.  The target sample is smaller but still substantial: **2,011,394 rows**, **54,729 inventors**, and **2,873 firms**.   The contrast itself is informative as acquiror events are broad integration shocks affecting many inventors, while target events are more concentrated and more likely to involve smaller organizations.

### Pre-period descriptives and what they imply

The pre-period balance diagnostics help calibrate both the economic magnitudes and the credibility of the matched comparison design.  The raw mean differences indicate that treated inventors are not mechanically identical to controls before the merger event. Acquiror-side treated inventors are stronger ex ante on productivity and value-weighted innovation measures: they average **0.95 patents per year** versus **0.74** for controls, **28.6 citations per year** versus **16.5**, and an average `xi_real` of about **16.7** versus **6.6**.  Target-side treated inventors are less distinct from controls in patenting levels, with **0.82 patents per year** versus **0.74**, but they are more mobile and somewhat more exploration-oriented: their pre-period move-year incidence is **1.9%** versus **0.9%** for controls, and their exploration share is about **24.1%** versus **21.8%**.  These differences are economically meaningful, but the standardized balance diagnostics show that they are moderate relative to the dispersion of inventor outcomes.


![Pre-treatment love plot: acquiror inventors vs. matched controls](../figures/love_plot_acquiror_vs_control_pre_period.png)

![Pre-treatment love plot: target inventors vs. matched controls](../figures/love_plot_target_vs_control_pre_period.png)

The love plots show that these raw differences are substantially attenuated in standardized terms. In the acquiror sample, most standardized mean differences are close to or below the conventional **±0.10** balance benchmark. The largest remaining imbalance is in `xi_real`, where treated acquiror inventors remain substantially above controls before treatment. In the target sample, all standardized mean differences remain below the **±0.10** benchmark, with the largest residual differences appearing for move-year incidence, citations, and exploration. Overall, the matched controls provide a broadly comparable pre-treatment comparison group, but the remaining pre-period differences make the dynamic pre-trend diagnostics especially important for assessing whether the parallel-trends assumption is plausible.

## 3. Headline inventor-year results

The inventor-year baseline results suggest two economically different mechanisms, but as the discussion below make clear the interpretation has to be measured.  Acquiror-side inventor, i.e., inventor who were working for the acquiring firm in an M&A transaction, reorient their innovation activity with increased exploration and labor mobility; target-side inventors who work for the acquired firm become less productive but only after some time delay, making the effects less clean.

### Acquiror inventors: reorientation, with visible pre-deal adjustment

For acquiror inventors, the preferred `nn1_x_firm` baseline estimates are strongest for search-direction and mobility outcomes:

- `exploration_inv`: **+0.019** with `p < 0.001`,
- `is_move_year`: **+0.008** with `p < 0.001`,
- `novelty_score_group`: **+0.004** with `p < 0.001`,
- `total_patents`: **-0.027** with `p = 0.134`, not significant,
- `cites`: **+2.177** with `p = 0.293`, not significant.

Acquiror inventors appear to be reallocated toward different kinds of inventive work.  The event-study figures support this broad interpretation, but not in a textbook “flat pre-period, sharp post-period jump” form.  The exploration path is V-shaped before the event: it is positive at `k=-5`, negative at `k=-4` to `k=-2`, then positive and persistent after the deal.  This pattern is consistent with reorientation around the deal window, but it also signals pre-deal adjustment or differential trajectories.

The mobility graph is even less suitable as a headline causal plot.  It shows a pronounced spike around `k=1`, followed by attenuation and later negative coefficients.  This is still economically useful: the run-up to acquisition and the subsequent integration may trigger short-run reassignment or mobility, followed by stabilization.  But because the pre-period slopes upward toward the event and the later post-period reverses, the mobility patterns is not a clean persistent effect.

### Target inventors: severe disruption, but weak dynamic identification

For target inventors, the preferred baseline estimates are more negative on output outcomes:

- `total_patents`: **-0.510** with `p = 0.029`,
- `cites`: **-22.121** with `p = 0.031`,
- `is_move_year`: **-0.012** with `p = 0.076`,
- `exploration_inv`: essentially zero and not significant.

The target inventor total-patents event study is shows a severe negative effect in the post period, especially in later years.  But it also shows negative coefficients already throughout the pre-period.  This is exactly the kind of plot that should make us cautious: acquired target inventors may already be on weaker output trajectories before the acquisition.  The target estimates are therefore best framed as **disruption/selection evidence around acquired firms**, not as a clean causal evidence given the delayed effects.


### Figure: inventor-year event-study 

The compact evidence map below is useful as a first directional summary:  The acquiror side has significant positive effects for exploration, mobility, and novelty, while the target side has significant negative effects for patents and citations.  Thus, acquiror inventors move on behavior/search outcomes; target inventors move on output outcomes. 

![Preferred inventor-year evidence map](../figures/generated/inventor_preferred_evidence_map_v2.png)



The acquiror side event study plot for exploration is the most useful inventor-year mechanism plot, but it is not a perfect parallel-trends figure.  The path is positive at `k=-5`, turns negative in the middle pre-period, and then becomes positive after the event.  The post-event coefficients are persistently positive, which supports the exploration/reorientation interpretation.  The V-shaped pre-period means the effect should be described as reorientation around the M&A window, not as a clean discrete jump exactly at closing.  However, this is a very cautious interpretation: five years of pre and post period are very long, and a shorted window would show cleaner effects.  

![Acquiror inventor exploration event study](../figures/inventor_year_event_study/es_inv_year_acquiror_vs_control_nn1_x_firm_exploration_inv.png)



The inventor mobility event study shows that innovation is not detached from labor reallocation.  However, the visual pattern is not a persistent mobility increase: the coefficient spikes around `k=1`, then turns negative by later post-event years.  Economically, this fits with rising labor mobility in the build-up to the merger and a short-run reallocation or integration shock followed by stabilization.  Econometrically, the pre-period slope and later reversal mean this figure more confirms a mechanism rather than a clean pre-trend free identification patterns.  Again, however, looking at just a few years before and after the merger, the effect become more clear.

![Acquiror inventor mobility event study](../figures/inventor_year_event_study/es_inv_year_acquiror_vs_control_nn1_x_firm_is_move_year.png)



The effects on patenting for inventors working for targeted firms are important, but also show that results should be viewed with caution.  The later post-period decline is large, but the pre-period coefficients are already negative.  That means target inventors do not look like they were tracking controls closely before the deal.  The plot is therefore best interpreted as evidence that target inventors are on a disruption or selection trajectory around acquisitions, not as clean dynamic proof that the acquisition alone caused the late post-period decline in patenting.

![Target inventor patents event study](../figures/inventor_year_event_study/es_inv_year_target_vs_control_nn1_x_firm_total_patents.png)


## 4. Dynamic DiD methods: why staggered timing matters

The baseline DiD is useful but incomplete because M&A events occur in different years.  With staggered treatment timing, a simple two-way fixed-effects event study can mix cohorts and comparison groups in ways that are hard to interpret.  The project therefore implements modern dynamic estimators.

### Callaway--Sant'Anna / CSDID

The CSDID estimator focuses on group-time treatment effects:

$$
ATT(g,t) = E[Y_t(1)-Y_t(0) \mid G=g],
$$

where $G=g$ is the cohort first treated in year $g$.  This measures the average treatment effect for cohort $g$ in calendar year $t$, relative to the counterfactual outcome that cohort would have had absent treatment.  The counterfactual outcome $Y_t(0)$ is estimated by comparing each treated cohort to units that are either not yet treated or never treated in calendar year $t$, using a doubly robust estimator with covariates when available and an inverse-probability-weighted fallback otherwise.  The doubly robust CSDID specification estimates the untreated counterfactual by combining outcome regression with inverse-probability weighting: it adjusts for observed covariates directly while also reweighting comparison units to resemble the treated cohort. Note that this is robust as the $ATT(g,t)$ remains consistently estimated if either the outcome model or the treatment-assignment model is correctly specified.  After estimating $ATT(g,t)$, the implementation maps calendar time into event time $e=t-g$ and aggregates effects using cohort-size weights.:

$$
\widehat{ATT}(e)
=
\sum_{g \in \mathcal{G}_e}
w_{g,e}\widehat{ATT}(g,g+e),
$$

where

$$
w_{g,e}
=
\frac{N_g}{\sum_{g' \in \mathcal{G}_e} N_{g'}}.
$$

Here, $\mathcal{G}_e$ is the set of treated cohorts observed at event time $e$, and $N_g$ is the number of treated inventors in cohort $g$. Thus, cohorts with more treated inventors receive more weight when aggregating the group-time effects into the dynamic event-study path.


For the CSDID dynamic aggregation, the program compute approximate uncertainty using a lightweight bootstrap over treated inventors.  In each bootstrap draw $b$, treated inventor IDs are sampled with replacement, producing bootstrap cohort counts $N_g^{(b)}$ and weights

$$
w_{g,e}^{(b)}
=
\frac{N_g^{(b)}}{\sum_{g' \in \mathcal{G}_e} N_{g'}^{(b)}}.
$$

The bootstrap event-time estimate is then

$$
\widehat{ATT}^{(b)}(e)
=
\sum_{g \in \mathcal{G}_e}
w_{g,e}^{(b)}
\widehat{ATT}(g,g+e),
$$

and the reported standard error is the standard deviation of $\widehat{ATT}^{(b)}(e)$ across bootstrap draws. This procedure is computationally much cheaper than repeatedly refitting the full CSDID estimator on large inventor-year panels, but it should be interpreted as uncertainty from the cohort-weighted aggregation step rather than as a full bootstrap of the underlying $ATT(g,t)$ estimates.

For this reason, the combined Sun--Abraham/CSDID comparison figure reports the CSDID point estimates without confidence intervals. Adding the bootstrap intervals could give a misleading impression of precision because the intervals reflect variation in cohort composition from the aggregation bootstrap, not the full first-stage uncertainty from estimating each group-time treatment effect.


### Sun--Abraham interaction-weighted event study

The Sun--Abraham estimator builds cohort-by-event-time indicators and then aggregates them with cohort weights:

$$
Y_{it}=\sum_{g}\sum_{k
\neq -1}	\beta_{gk} \cdot 1\{G_i=g\}1\{t-g=k\}+X_{it-1}'\delta+\alpha_i+\lambda_t+
\epsilon_{it}.
$$

Here, $G_i$ denotes unit $i$'s treatment cohort, $k=t-g$ is event time relative to treatment.  The cohort-by-event-time coefficients $\beta_{gk}$ are then aggregated into treatment effects for each event time $k$. The interaction-weighted estimate is

$$
\hat\theta_k
=
\sum_{g \in \mathcal{G}_k}
w_{gk}\hat\beta_{gk},
\qquad
w_{gk}
=
\frac{N_g}{\sum_{g' \in \mathcal{G}_k}N_{g'}},
$$

where $\mathcal{G}_k$ is the set of treatment cohorts observed at event time $k$, and $N_g$ is the number of unique treated units in cohort $g$. Thus, each event-time coefficient is a cohort-size-weighted average of the cohort-specific treatment effects available at that relative year.

This directly addresses treatment-effect heterogeneity across cohorts.  It is especially useful as a robustness check for whether the baseline event-study pattern is driven by staggered-adoption artifacts.


### Figure: dynamic-estimator robustness checks

The Sun--Abraham exploration figure is the strongest dynamic robustness plot for acquiror inventors.  It shows positive post-event exploration estimates from `k=0` onward, while still exposing some pre-period behavior that is marginally insignificant, with the effect at `k=-5` as a positive outlier.  This supports the acquiror reorientation story, but with the same caveat as the baseline event study: timing is not perfectly sharp.

![Sun-Abraham acquiror exploration](../figures/inventor_year_event_study/advanced/sa_inv_year_acquiror_vs_control_nn1_x_firm_exploration_inv_plot.png)



The CSDID figures are more complicated.  Acquiror exploration is positive at event time 0 but turns negative in event years 1--3.  Acquiror mobility also spikes at event time 0 and then becomes negative.  These CSDID plots do not cleanly reinforce the baseline story.  They are useful diagnostics because they show that the inferred dynamic pattern is estimator-sensitive.  Not all modern DiD estimators tell the same story and there is no universally stable positive treatment path.  A more accurate statement is that Sun--Abraham supports the acquiror exploration mechanism, while CSDID points to short-run adjustment followed by negative later dynamics.


![Acquiror exploration across dynamic estimators](../figures/generated/acquiror_exploration_dynamic_methods.png)


Note in this context that a negative CSDID estimate that differs from the Sun--Abraham result may reflect the greater support requirements and implementation fragility of the CSDID estimator rather than a clean contradiction in the underlying economic effect.  In this inventor-year setting, CSDID relies on identified cohort-time cells with valid not-yet-treated or never-treated controls, drops small or unsupported cells, uses a shorter event window, and then aggregates only the remaining $ATT(g,t)$ estimates, so a negative dynamic effect can be driven by a more selective and noisier subset of cohorts than the broader Sun--Abraham interaction-weighted estimate.


## 5. Heterogeneity: where effects are strongest

The project also estimates triple-difference specifications asking whether M&A effects vary across inventor or firm types:

$$
Y_{it} = \beta_1\text{PostTreat}_{it} + \beta_2(\text{Post}_{it}\times Z_i)
       + \beta_3(\text{PostTreat}_{it}\times Z_i) + X_{it-1}'\delta + \alpha_i + \lambda_t + \varepsilon_{it}.
$$

Here $Z_i$ can be a measure of firm size, relative deal size, or an inventor's within-firm rank based on cumulative patents or citations.  The coefficient $\beta_3$ shows whether the treatment effect differs for units with characteristic $Z_i$. For binary heterogeneity variables, such as being in the upper half of inventors within the same firm, $\beta_1$ is the estimated post-treatment effect for the lower-rank group and $\beta_1+\beta_3$ is the corresponding effect for the upper-rank group.

The most interpretable inventor heterogeneity split is the within-firm inventor-rank measure, i.e., whether the inventor was ranked in the upper half of inventors working for the same firm at `t=-1`, based on cumulative innovation output.  For acquiror inventors, the high-cumulative-patent interaction is negative for productivity and mobility outcomes but positive for exploration. The baseline effect for lower-rank acquiror inventors is **+0.40 patents**, **+29.64 citations**, and **+2.0 percentage points in move-year incidence**. The interaction for upper-rank inventors is **-0.48 patents**, **-23.28 citations**, and **-1.84 percentage points in move-year incidence**. Thus, the implied effect for upper-rank acquiror inventors is much smaller: about **-0.08 patents**, **+6.36 citations**, and only **+0.16 percentage points in move-year incidence**.

This pattern suggests that the acquiror-side effects are concentrated among lower-rank inventors rather than among already-established inventors as patenting, citations, and mobility increase more after treatment for those less productive investors.  The exploration result, however, points in a different direction: lower-rank acquiror inventors have an exploration effect of about **-2.2 percentage points**, while the upper-rank interaction is **+3.1 percentage points**, implying a small positive net exploration effect of roughly **+0.9 percentage points** for upper-rank inventors.  Economically, this is consistent with a more reallocation of less-established inventors and more exploratory redeployment of established inventors.

For targets, heterogeneity is harder to summarize as one clean mechanism, but the within-firm inventor-rank split again provides the most interpretable pattern.  For lower-rank target inventors, the baseline effects are **+0.75 patents** and **+35.88 citations**.  The upper-rank interactions are sharply negative: **-1.63 patents** and **-72.57 citations**, implying net effects for upper-rank target inventors of about **-0.88 patents** and **-36.70 citations**.  This suggests that, on the target side, any positive productivity response is concentrated among lower-rank inventors, while higher-rank target inventors experience weaker or negative post-treatment productivity effects.


### Figure: triple-DiD heterogeneity event studies

The following figure shows the the triple interaction for acquiror inventors with the heterogeneity split based on on whether an inventor is in the upper half of cumulative patents within the firm at baseline `t=-1`.  In these heterogeneity event-study figures, the reference-group path reports the event-time coefficients for inventors with $Z_i=0$, while the upper-group path reports the combined effect for inventors with $Z_i=1$, i.e. the base event-time coefficient plus the interaction coefficient.  Thus, the difference between the two plotted paths is the triple-difference heterogeneity component.

The high-baseline-patent group shows a steadily positive post-event exploration path, while the reference group declines after the event.  This suggests that more established inventors appear better positioned to redirect search after M&A, while the reference group of less productive inventors in the pre-period may face more disruption limiting exploration.  The caveat is that the pre-period is not perfectly parallel, especially for the reference group, which shows a slight positive pre-trend.  While this limits the degree to which the effects can be interpreted as standalone causal estimates, it supports the idea that the treatment effects of mergers are not uniform.

![Triple-DiD acquiror exploration by inventor rank](../figures/inventor_year_event_study/es_inv_year_acquiror_vs_control_nn1_x_firm_ddd_Z_upper_half_cum_patents_within_firm_exploration_inv.png)



The mobility heterogeneity plot is less clean than the exploration heterogeneity plot.  The coefficients for the reference group have a strong positive slope in the pre-period, turn significant and positive in `t=0` and `t=1`, and then begin to fall again and turn negative.  The upper-half group is more stable and has treatment effects peak in `t=1`.  This suggests that mobility/reassignment is more concentrated among less established inventors in the references group, while established inventors are less displaced.  Econometrically, the pre-period movement is substantial, so this figure should be treated as mechanism exploration rather than a clean causal estimate.  Especially the rising level of inventor mobility for the reference group in the pre-period could hint at a broader mechanism that increases the likelihood of mergers as younger and less productive inventors increasingly leave the firm at the same time.

![Triple-DiD acquiror mobility by inventor rank](../figures/inventor_year_event_study/es_inv_year_acquiror_vs_control_nn1_x_firm_ddd_Z_upper_half_cum_patents_within_firm_is_move_year.png)


## 6. Firm-level results for aggregate context

The firm-level analysis asks whether the inventor-level effects aggregate into firm-level innovation outcomes.  This analysis uses role-specific acquiror and target panels, matches treated firms to non-M&A controls using pre-event size and industry information, and estimates the same fixed-effects DiD and event-study designs as used for the inventor-level analysis.  Overall, the results provide context and robustness for the inventor-level outcomes.

For matching, the procedure constructs cohort-specific stacked panels by matching treated firms at `t=−1` to same-sic3 control firms using Mahalanobis distance on `log_sale` and `log_mv`.  This matching step before the actual empirical analysis matters.  In the acquiror sample, treated firms are much larger before matching: the raw pre-match difference is **1.69 log sales** and **1.95 log market value**.  After matching to control firm, those gaps fall to **0.14 log sales** and **0.22 log market value**.  Target firms were already closer to controls on these size variables.  This does not prove identification, but it makes the parallel trends assumptions more plausible if treated and control firms are matched based on industry and pre-treatment size and reduces the risk that any treatment effects are actually related to size-specific factors.

Note that outcome for the firm-level panel are measured differently for interpretability and distributional reasons than in the inventor panel: firm-year innovation outcomes are often highly skewed aggregates of count variables across inventors, so most specification will use a log-transformation, $\log(1+y)$, to reduces the influence of extreme values and makes coefficients closer to semi-elasticities, i.e., a coefficient $\beta$ can be interpreted approximately as a $100 \times \beta$% change in the outcome.  Inventor-year outcomes, on the other hand, are kept in original units so coefficients can be read as changes in patents, citations, exploration shares, or move-year probabilities per inventor-year.

The firm-level baseline estimates show negative post-M&A effects for several aggregate acquiror outcomes, including the log number of patents, `log1p_total_patents` (**-0.058**, `p = 0.012`), the log aggregate KPSS patent-value measure, `log1p_xi_real` (**-0.122**, `p = 0.001`), and the log number of unique inventors, `log1p_num_inventors` (**-0.070**, `p = 0.004`).  Target firms show larger negative aggregate baseline effects, including `log1p_total_patents` (**-0.140**, `p < 0.001`), `log1p_cites` (**-0.370**, `p < 0.001`), and `log1p_num_inventors` (**-0.169**, `p < 0.001`).

The event-study plots, however, are not clean enough to make the firm-level effects the headline results.  Target-firm total patents show a strong positive pre-period path that falls toward zero around the event.  This is a severe pre-trend, which could indicate that firms with weakening innovation trajectories are more likely to become acquisition targets, or that target firms experience changes before the deal is announced in anticipation.  The decline in aggregate patenting is broadly consistent with the decline in patenting at the inventor level, with the caveat that the aggregate firm-level effects begin trending downward earlier, potentially because inventors leave targeted firms before the M&A event or because innovative activity is already being reorganized before completion. There is also selection in the target-firm panel: not every target continues to exist as an independent observable firm after the merger, while inventor outcomes can still be observed if inventors move to another company.  Acquiror firm total patents also drift down before and after the event (though less extreme than for target firm), with wide uncertainty bands, consistent with the insignificant treatment effects for the inventor-year panel.  These patterns are informative about aggregate reorganization and selection, but they are weaker support for a causal effect of M&As than the inventor-year mechanism.


### Figure: firm-level context and diagnostics


The matching balance before and after matching figure shows why the matched stacked design is preferable to an unmatched firm comparison.  For acquirors, the raw treated-control size gap is very large and matching materially reduces it.  For targets, treated and control firms are already closer on these size measures.  The results, however, only cover only the size variables used for matching, not all innovation-history or technology variables.

![Firm matching balance before and after matching](../figures/generated/firm_balance_pre_post_matching_v2.png)


The following figure summarizes the baseline DiD estimation results.  The selected outcomes show a broadly negative aggregate pattern for both acquirors and targets, with especially large declines for target firms in citations, innovation value, total patents, and the number of inventors.  For acquirors, the point estimates are also mostly negative, but generally smaller in magnitude.  Economically, this supports the interpretation that firm-level innovation output and inventor counts weaken after M&A, while the inventor-year evidence points more specifically to reorientation, mobility, and heterogeneous inventor-level responses.  There is, however, not a one-to-one mapping between firm- and inventor-level outcomes: inventor-level outcomes can still be observed if an inventor moves to a different firm, while aggregate firm-level outcomes are tied to the patenting activity assigned to the original acquiror or target firm.  Furthermore, target firm-level outcomes are conditional on the target remaining observable after the merger, which makes the post-M&A target-firm panel especially selected.

![Firm-level selected baseline summary](../figures/generated/firm_baseline_selected_summary_v2.png)


The event study plot shows that for acquiror firm the total-patenting path is downward sloping over the event window, but uncertainty is large and the pre-period is not flat.   While this firm-level graph is not the cleanest causal evidence, its result is consistent with the insignificant results for the inventor-level patenting outcomes for acquiror inventor and supports that treatments effects are more likely to be related to changes in innovation direction rather than total output.

![Acquiror firm total patents event study](../figures/es_acquiror_log1p_total_patents.png)


The event-study plot for total patenting of target firms shows that the pre-period coefficients are large, positive, and declining toward the omitted period before the deal.  Post-period estimates are closer to zero and generally have larger error bands.  Economically, this is consistent with targets being selected after an earlier high-innovation period or entering the acquisition window on a declining trajectory. Econometrically, this is not a clean parallel-trends design.  It nevertheless provides useful context: the decline in innovation output for target inventors is also reflected at the aggregate firm level, while the earlier start of the firm-level decline is consistent with inventors leaving target firms before the merger or innovation activity being reorganized anticipating the merger.

![Target firm total patents event study](../figures/es_target_log1p_total_patents.png)


## 7. Additional implemented methods and how to interpret them

A strength of the project is that it implements several estimators beyond the baseline matched stacked DiD and event-study design.  These additional methods ask whether the main patenting pattern is visible for different empirical approaches.  However, they are best interpreted as robustness and diagnostic evidence rather than headline results.  The main empirical story should continue to rely on the matched stacked DiD/event-study estimates and the inventor-level evidence, while the methods below provide additional context through robustness, diagnostics, and economic interpretation.

### BJS imputation

The [Borusyak, Jaravel, and Spiess (BJS)](https://academic.oup.com/restud/article/91/6/3253/7601390) imputation estimates what treated firms would have looked like absent treatment.  Specifically, the untreated outcome model is estimated only on observations with $D_{it}=0$, i.e., control firms that were never treated and pre-period observations for treated firms:

$$
Y_{it}=\alpha_i+\lambda_t+X_{it}'\gamma+u_{it}, \qquad D_{it}=0,
$$

where $\alpha_i$ are firm fixed effects, $\lambda_t$ are year fixed effects, and $X_{it}$ are the same controls used in the firm-level analysis.  The fitted untreated potential outcome is

$$
\widehat{Y}_{it}(0)=\widehat{\alpha}_i+\widehat{\lambda}_t+X_{it}'\widehat{\gamma}.
$$

For treated post-event observations, the imputed treatment effect is

$$
\widehat{\tau}_{it}=Y_{it}-\widehat{Y}_{it}(0).
$$

Dynamic effects are then averaged by event time:

$$
\widehat{ATT}_k
=
\frac{1}{N_k}
\sum_{i,t:\,t-G_i=k,\,D_{it}=1}
\widehat{\tau}_{it}.
$$

This method has the advantage that treated firms are compared to an estimated untreated path, rather than only to control firms inside a two-way fixed-effects regression.  Standard errors and confidence intervals are estimated using bootstrap as the closed form solution for the variance of the estimator is quite complex.  For the firm-level BJS results, the default inference routine uses a cluster bootstrap at the firm level.  In each bootstrap draw, the code samples firm identifiers with replacement, keeps all observations for each sampled firm, re-estimates the untreated-outcome imputation model on the resampled data, recomputes the residualized treatment effects, and then re-averages those effects by event time.  The implementation also contains an alternative wild-bootstrap-style option that perturbs centered firm-level contributions without refitting the first-stage imputation model; this is mainly useful for larger panels where repeatedly refitting the model would be expensive.

For total patenting, the acquiror BJS path is negative after the event, with estimates of about **-0.28** at `k=0`, **-0.27** at `k=1`, and **-0.10** by `k=5`.  This is directionally consistent with decreased post-M&A patenting weakness, but the confidence intervals are wide and mostly include zero.  The target BJS path is less aligned with the main story: post-event estimates are positive after the event.  This does not invalidate the project’s broader interpretation, but it is a useful warning that target firm-level patenting is sensitive to how the counterfactual path is constructed.  


### Synthetic control

The stacked synthetic-control estimator builds a firm-specific comparison path.  For each eligible treated firm, the method chooses donor weights $w_j$ for control firms so that the weighted donor path matches the treated firm’s pre-event outcome path:

$$
\widehat{Y}^{SC}_{it}
=
\widehat{\alpha}_i
+
\sum_{j\in\mathcal{D}_i}
\widehat{w}_{ij}Y_{jt},
\qquad
\widehat{w}_{ij}\geq 0.
$$

The treatment gap is the difference between the treated firm and its synthetic control unit:

$$
\widehat{\tau}^{SC}_{it}
=
Y_{it}
-
\widehat{Y}^{SC}_{it}.
$$

The stacked event-study path averages these firm-level gaps by relative year:

$$
\widehat{ATT}^{SC}_k
=
\frac{1}{N_k}
\sum_{i,t:\,t-G_i=k}
\left(Y_{it}-\widehat{Y}^{SC}_{it}\right).
$$

The project implementation uses a ridge-regularized nonnegative least-squares synthetic-control procedure.  It places more weight on pre-treatment years closer to the M&A event, estimates treated-minus-synthetic gaps, aggregates those gaps by event time, and bootstraps over treated firms to estimate confidence intervals, i.e., treated firms are resampled with replacement and the mean event-time gap is recomputed in each bootstrap draw.  This makes SCM useful by itself because it gives a visually intuitive counterfactual: if the synthetic-control design is credible, pre-event gaps should be close to zero, and post-event deviations show how treated firms diverge from similar donor-based paths.

For total patenting, the SCM results are the most supportive of the additional methods. For acquirors, 648 of 682 eligible treated firms have successful synthetic-control fits, and the average post-event effect over `k\in[0,5]` is **-0.0695**, with a 95% confidence interval of **[-0.1219,-0.0124]**.  For targets, 115 of 144 eligible treated firms have successful fits, and the average post-event effect is **-0.1011**, with a 95% confidence interval of **[-0.1904,-0.0102]**.  In both cases, pre-event gaps are close to zero and post-event gaps turn negative. This supports the economic interpretation that treated firms patent less than comparable synthetic controls after M&A and that this effect is strongest for target firms. 


### Causal forest

The causal forest is designed to study heterogeneity.  The project converts the panel into a cross-sectional treatment-effect setting. Pre-treatment covariates are measured at `k=-1` and the outcome is the average post-event outcome over a short window.  The objective is to estimate the conditional average treatment effect,

$$
\tau(x)
=
E\!\left[Y_i(1)-Y_i(0)\mid X_i=x\right],
$$

where $X_i$ are the pre-treatment firm covariates and $x$ is a particular covariate profile.

In the project implementation, the causal forest uses `CausalForestDML`.  Random forests are used as nuisance models for $E[Y_i\mid X_i]$ and $E[D_i\mid X_i]$, so the treatment-effect model is estimated after residualizing both the post-period outcome and the treatment indicator with respect to pre-period firm covariates.  The final forest estimates conditional treatment effects $\tau(X_i)$, reports their average as the ATE, computes an ATE confidence interval, saves firm-level CATEs, and reports feature importances that summarize which covariates are most important for explaining heterogeneous treatment effects.  This is useful because average DiD effects can hide substantial variation across firms. M&A may reduce patenting for some firms, increase it for others, and have little effect for many.  Causal forest helps identify which pre-treatment characteristics are most associated with that variation.

For log total patents, the causal-forest ATEs are not precise.  The acquiror ATE is **0.0879**, with a 95% confidence interval of **[-0.3803,0.5562]**.  The target ATE is **0.0120**, with a 95% confidence interval of **[-0.3780,0.4020]**.  Thus, the intervals are too wide for a useful interpretation of ATE.  Note als that the causal-forest ATE is not directly comparable to the main DiD coefficient. The main DiD estimate is a panel fixed-effects pre/post treatment effect: it uses within-firm changes around the M&A event relative to matched controls and absorbs firm and year fixed effects. The causal forest instead instead is a cross-sectional heterogeneity exercise, using pre-period firm covariates and average post-period outcomes. As a result, its ATE can be statistically insignificant even when the main DiD coefficient is significantly negative.  The most useful result is the heterogeneity ranking.  Lagged log sales is the most important heterogeneity variable for both acquirors and targets, suggesting that firm size is central to how patenting responds to M&A.   

This causal-forest heterogeneity result is reflected in the log-sales triple-DiD specifications:  firm size is not just predictive in the forest, but also appears in the interaction regressions as economically meaningful treatment-effect variation.  For acquirors, the continuous log-sales interaction is positive and marginally significant (`Post_Treated_Z_log_sales_cont = 0.0225`, `p=0.063`), and the five-bin specification shows a significant interaction for the second size bin; for targets, the interaction is negative in the larger-size bins, with the three-bin specification showing a significant negative interaction for the largest size group (`Post_Treated_Z_log_sales_q3_2 = -0.1452`, \(p=0.049\)).

The log-sales triple-DiD results are consistent with the causal-forest interpretation that firm size matters for treatment-effect heterogeneity.  A useful structured heterogeneity check is the target-firm Triple-DiD specification using three baseline log-sales bins in `t=-1`. The specification can be written as

$$
Y_{it}
=
\beta \, \text{PostTreated}_{it}
+
\sum_{m=1}^{2}
\delta_m \left(\text{Post}_{it}\times Z_{im}\right)
+
\sum_{m=1}^{2}
\theta_m \left(\text{PostTreated}_{it}\times Z_{im}\right)
+
X_{it}'\gamma
+
\alpha_i
+
\lambda_t
+
\varepsilon_{it},
$$

where $Z_{im}$ are indicators for the middle and largest baseline log-sales terciles, with the smallest tercile omitted.  The coefficient $\beta$ gives the post-M&A effect for the smallest target firms, while $\theta_m$ measures how much the treatment effect differs for larger target firms relative to that smallest-size group.

For log total patents, the smallest target firms have a negative post-M&A effect of **-0.104** (**p=0.022**).  The interaction for the largest size tercile is also negative and statistically significant: **\theta_2=-0.145** (**p=0.049**).  This means that the estimated post-M&A patenting effect for the largest target firms is approximately **-0.104 - 0.145 = -0.249.**

Economically, this suggests that target-side patenting declines are stronger among larger targets. This is consistent with the causal-forest result that lagged log sales is the most important heterogeneity variable.  However, this is still supporting evidence rather than a headline result, because it is one heterogeneity specification and the pattern is not equally strong across all bin definitions.


### Placebos and diagnostics

The placebo routines test whether the empirical design generates effects when treatment timing should not have a causal interpretation. Here, $G_i$ is the true M&A event year for treated firm $i$. In the true design, the post-treatment indicator turns on in and after the actual event year:

$$
D_{it}=1\{t\geq G_i\}.
$$

The lead placebo instead assigns the same firm a fake treatment year that occurs $L$ years before the true deal:

$$
G_i^{lead}=G_i-L,
\qquad
D_{it}^{lead}=1\{t\geq G_i^{lead}\}.
$$

For example, if a firm is actually acquired in 2005 and $L=3$, the placebo treatment year is 2002. The placebo regression then asks whether the model already finds an “effect” before the actual M&A event. A strong placebo effect would suggest that the treated firms were already on a different trajectory before treatment.

The permutation placebo keeps the set of treatment years but randomly reallocates them across treated firms:

$$
G_i^{perm}=\pi(G_i),
\qquad
D_{it}^{perm}=1\{t\geq G_i^{perm}\}.
$$

Here, $\pi(G_i)$ denotes a random permutation of the actual cohort years.  This preserves the overall distribution of treatment timing but breaks the link between a specific firm and its actual M&A year.  The placebo regression then asks whether similar effects appear when treatment timing is artificially reassigned.  If the true estimates are stronger and more economically coherent than the permuted estimates, this would supports the interpretation that the main results are connected to actual M&A timing rather than generic time patterns or features of the model specification.

Overall, the tests probe whether estimated effects are mechanically produced by the model, by persistent pre-trends, or by timing artifacts.  Thus, they should be seen as design diagnostics.  In the project implementation, placebo timing is fed back through the same baseline and advanced routines, including DiD/event-study, Sun-Abraham, and BJS versions for significant firm-level outcomes.  For the patenting results, the target placebo paths are generally small and mixed, which is reassuring.  The acquiror placebo paths are less clean, suggesting that acquiror firm-level patenting is more sensitive to timing and specification choices. 

### Figure: robustness and diagnostic visualizations

The target BJS plot is not a clean reinforcement of the baseline target decline; instead, it shows positive post-event imputed effects with wide confidence intervals and visible pre-period residual movement.  Overall, while innovation outputs for target firm are disrupted in several baseline specifications, the precise dynamic pattern is fragile across estimators.

![BJS target total patents](../figures/advanced/bjs_target_total_patents_plot.png)


The SCM plots are more visually interpretable than BJS.  The target SCM figure shows the pre-period treated-minus-synthetic gap is close to zero and the post-period path turns negative.  However, the confidence interval widens substantially after the event, so the figure supports direction and intuition more than precise inference.

![Stacked SCM target total patents](../figures/advanced/scm_target_log1p_total_patents_eventstudy_ci.png)


The causal-forest evidence is useful for heterogeneity.  The acquiror distribution is wider and has a longer positive tail, with an average CATE around 0.088.  The target distribution is centered much closer to zero, with an average CATE around 0.012.  This suggests that average firm-level effects hide substantial cross-firm dispersion, especially among acquirors.

![Causal-forest CATE comparison](../figures/generated/causal_forest_cates_comparison.png)



## 8. Interpretation of the full evidence

The most coherent economic interpretation is that M&A changes the innovation performance and direction, but the strength of the evidence differs across roles and outcomes.  Results for acquiror inventors show search-direction changes, while patenting output is disrupted for target inventors, although the evidence is less cleanly identified for targets.


For **acquirors**, the inventor-level evidence points to reorientation.  Exploration increases in the post-event period in both the baseline event study and Sun--Abraham plot.  The mobility plot suggests a short-run reallocation spike rather than a persistent increase in mobility.  The visual evidence is not perfectly clean because several acquiror paths move before the omitted year, but the repeated pattern across search-direction and heterogeneity results makes acquiror reorientation the best-supported mechanism.

For **targets**, the evidence points more toward disruption and potential selection.  Target inventor patents fall sharply in later post-event years, and target firm-level baseline estimates are negative.  But the target event studies show substantial pre-period movement.  This means that target results are economically severe but dynamically less clean: acquired targets appear to be on different innovation trajectories even before the acquisition, which could contribute to them being eligible targets in the first place.

The evidence on **aggregate firm** is contextual.  Firm-level event studies show pre-trends and wide error bands, while SCM and BJS provide mixed but useful robustness diagnostics supporting the general story from the inventor-level analysis.  These aggregate plots show that M&A can depress patenting at the firm level even when inventor-level mechanisms are more nuanced.  

